# Does local visual context improve Shape recognition?

We train Patchify, Pointwise and Conv Stem with identical questions, update budgets and downstream ViT computations. Pointwise controls for additional capacity without neighboring-patch mixing. The comparison tests whether local context helps the model learn Shape and use that evidence in the other tasks.

In [1]:
from pathlib import Path
import sys, json, math, hashlib
from collections import Counter
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score,balanced_accuracy_score
from scipy.stats import t as student_t
from IPython.display import display
from src.medium_visual_world.learning import AttributeBindingViT, MediumDataset, set_seed, environment
from src.medium_visual_world.generate import read_jsonl
from src.medium_visual_world.representation_probing import oracle, predict_rows

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(4,torch.get_num_threads()))
SEEDS = (42,43,44)
STEMS = ("patchify","pointwise","conv_stem")
LR,BATCH_SIZE,WEIGHT_DECAY,CLIP_NORM = 1e-4,50,1e-4,1.0
DATA_DIR = PROJECT_ROOT/"outputs/medium_world/dataset"
OUT_DIR = PROJECT_ROOT/"outputs/medium_world/experiments/architecture_protocol"
RETRAIN_ALL_STEMS=False  # True forces new training; otherwise only incomplete runs are trained
train_data=MediumDataset(DATA_DIR/"train.jsonl")
S_ROWS={family:list(read_jsonl(OUT_DIR/"selection"/(family+".jsonl"))) for family in
        ("conjunction","binding","swaps","operator","recolor","ood")}
COLOR_ROWS=[]
for r in S_ROWS["conjunction"]:
    rc={**r,"example_id":r["example_id"]+"|color_control","task":"color","group":"color_control"}
    rc["label"] = oracle(rc["scene"],"color",rc["color"],rc["shape"])
    COLOR_ROWS.append(rc)
QUERY_SHAPE_COUNT={r["example_id"]:sum(o["shape"]==r["shape"] for o in r["scene"]["objects"])
                   for r in S_ROWS["recolor"]}
ALL_S_ROWS=[*COLOR_ROWS,*[r for rows in S_ROWS.values() for r in rows]]
print("Selection questions:",len(ALL_S_ROWS),"device:",DEVICE)

Selection questions: 10500 device: cuda


## 1. Compare three visual stems

Each stem produces the same 16×16 grid of 48-dimensional tokens. Pointwise adds nonlinear processing within individual patches; Conv Stem uses overlapping 3×3 kernels. Their comparison tests the usefulness of local context, although the two stems also differ in depth and stride arrangement.

In [2]:
class PointwiseStemViT(AttributeBindingViT):
    def __init__(self,**kw):
        super().__init__(**kw)
        self.patch=nn.Sequential(nn.Conv2d(3,304,4,stride=4),nn.GELU(),
                                 nn.Conv2d(304,48,1),nn.GELU(),nn.Conv2d(48,48,1))

class ConvStemViT(AttributeBindingViT):
    def __init__(self,**kw):
        super().__init__(**kw)
        self.patch=nn.Sequential(nn.Conv2d(3,24,3,padding=1),nn.GELU(),
                                 nn.Conv2d(24,48,3,stride=2,padding=1),nn.GELU(),
                                 nn.Conv2d(48,48,3,stride=2,padding=1))

MODEL_CLASSES={"patchify":AttributeBindingViT,"pointwise":PointwiseStemViT,"conv_stem":ConvStemViT}

def build_model(stem):
    return MODEL_CLASSES[stem](dim=48,patch_size=4,depth=1)

@torch.no_grad()
def cpu_profile_flops(model):
    from torch.profiler import profile,ProfilerActivity
    model=model.cpu().eval()
    with profile(activities=[ProfilerActivity.CPU],with_flops=True) as prof:
        model(torch.rand(1,3,64,64),torch.tensor([[1,2,7,12]],dtype=torch.long))
    return int(sum(e.flops for e in prof.key_averages()))

specs=[]
for stem in STEMS:
    model=build_model(stem)
    specs.append(dict(stem=stem,parameters=sum(p.numel() for p in model.parameters()),
                      patch_parameters=sum(p.numel() for p in model.patch.parameters()),
                      profiled_flops_partial=cpu_profile_flops(model)))
display(pd.DataFrame(specs))

,stem,parameters,patch_parameters,profiled_flops_partial
0,patchify,42018,2352,20965760
1,pointwise,71554,31888,35907968
2,conv_stem,71538,31872,56945024


## 2. Match learning opportunities

Each training question receives 45 presentations. The three stems share the same question order and image augmentations within a seed, so differences cannot be explained simply by one architecture seeing more examples.

In [3]:
def make_batches(seed,mode,updates):
    choices=np.arange(len(train_data),dtype=np.int32)
    if mode=="shape_only":choices=choices[np.array(train_data.tasks)=="shape"]
    n=updates*BATCH_SIZE
    rng=np.random.default_rng(seed)
    order=np.concatenate([rng.permutation(choices) for _ in range(n//len(choices))])
    return order.reshape(updates,BATCH_SIZE)

def augment_matched(x,seed,step):
    generator=torch.Generator(device="cpu").manual_seed(seed*1000000+step)
    choices=torch.rand((len(x),3),generator=generator,device="cpu").to(x.device)<.5
    for axis in (0,1):
        mask=choices[:,axis,None,None,None]
        x=torch.where(mask,x.flip(-1-axis),x)
    return torch.where(choices[:,2,None,None,None],x.transpose(-1,-2),x)

BATCHES={(seed,mode):make_batches(seed,mode,2250 if mode=="shape_only" else 9000)
         for seed in SEEDS for mode in ("shape_only","mixed")}
for (seed,mode),batches in BATCHES.items():
    values,counts=np.unique(batches,return_counts=True)
    assert len(set(counts))==1 and int(counts[0])==45
print("Per-question presentations: 45; batches fixed across stems.")

Per-question presentations: 45; batches fixed across stems.


## 3. Train and evaluate the models

Shape-only training isolates prerequisite acquisition; Mixed training examines whether that improvement carries into Conjunction and Binding. Each run saves intermediate weights, predictions and task-specific metrics. Existing complete runs can be evaluated again without repeating training.

In [4]:
@torch.inference_mode()
def selection_evaluate(model,mode,image_cache):
    model.eval()
    rows=S_ROWS["recolor"] if mode=="shape_only" else ALL_S_ROWS
    frame=predict_rows(model,rows,DEVICE,batch_size=80,image_cache=image_cache)
    shape=frame[frame.group=="recolor"]
    shape_ba=balanced_accuracy_score(shape.label,shape.prediction)
    recolor_correct=float(shape.groupby("pair_id").correct.min().mean())
    single=shape[(shape.case=="present")&(shape.example_id.map(QUERY_SHAPE_COUNT)==1)]
    metrics=dict(shape_balanced_accuracy=shape_ba,shape_recolor_both_correct=recolor_correct,
                 shape_recolor_both_correct_single_instance=float(single.groupby("pair_id").correct.min().mean()))
    if mode=="mixed":
        color=frame[frame.group=="color_control"]
        metrics["color_balanced_accuracy"]=balanced_accuracy_score(color.label,color.prediction)
        conj=frame[frame.group=="conjunction"]
        for label, sub in conj.groupby("case"):
            metrics["conjunction_"+label+"_correct"]=sub.correct.mean()
        metrics["conjunction_four_stratum_balanced"]=conj.groupby("case").correct.mean().mean()
        bind=frame[frame.group=="binding"]
        metrics["binding_auc"]=roc_auc_score(bind.label,bind.prob_yes)
        for near in ("close","far"):
            sub=bind[bind.adjacency==near]
            metrics["binding_auc_"+near]=roc_auc_score(sub.label,sub.prob_yes)
        op=frame[frame.group=="operator"]
        grouped=op.groupby(["case","pair_id"]).correct.min().reset_index()
        metrics["operator_three_case_joint"]=grouped.groupby("case").correct.mean().mean()
        swaps=frame[frame.group=="swaps"]
        for suffix,part in (("",swaps[swaps.heldout_created==0]),
                            ("_heldout_created",swaps[swaps.heldout_created==1])):
            metrics["swap_eight_correct"+suffix]=float(part.groupby("pair_id").correct.min().mean())
            pivot=part[part.task=="conjunction"].pivot(index="pair_id",columns="state",values="prediction")
            metrics["conjunction_swap_flip_rate"+suffix]=float((pivot.before!=pivot.after).mean())
            metrics["swap_pairs"+suffix]=int(part.pair_id.nunique())
        metrics["repaired_ood_binding_auc"]=roc_auc_score(frame.loc[frame.group=="ood","label"],
                                                           frame.loc[frame.group=="ood","prob_yes"])
    return metrics,frame

def train_stem(stem,seed,mode):
    updates = 2250 if mode=="shape_only" else 9000
    eval_points = (750,1500,2250) if mode=="shape_only" else (5400,7200,9000)
    folder=OUT_DIR/"selection_stems"/mode/f"{stem}_seed{seed}"
    folder.mkdir(parents=True,exist_ok=True)
    set_seed(seed)
    model = build_model(stem).to(DEVICE)
    torch.manual_seed(seed)  # same dropout stream for every stem; init consumed different CPU draws
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    batches = BATCHES[seed, mode]
    schedule_digest=hashlib.sha256(batches.tobytes()).hexdigest()
    hist=[]
    image_cache={}
    for step,indices in enumerate(batches,start=1):
        x,q,y=train_data.batch(indices,torch.device(DEVICE))
        x=augment_matched(x,seed,step)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(x,q)
        loss = F.cross_entropy(logits,y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),CLIP_NORM)
        optimizer.step()
        if step in eval_points:
            metrics,preds=selection_evaluate(model,mode,image_cache)
            rec=dict(stem=stem,seed=seed,mode=mode,step=step,train_loss=float(loss.item()),**metrics)
            hist.append(rec)
            preds.drop(columns=["scene"],errors="ignore").to_csv(folder/f"selection_predictions_step{step}.csv",index=False)
            torch.save({"model":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                        "stem":stem,"seed":seed,"mode":mode,"step":step},folder/f"model_step{step}.pt")
            print(mode,stem,seed,step,"Shape BA",round(metrics["shape_balanced_accuracy"],4),
                  "recolor both",round(metrics["shape_recolor_both_correct"],4),flush=True)
    pd.DataFrame(hist).to_csv(folder/"selection_history.csv",index=False)
    settings=dict(stem=stem,seed=seed,mode=mode,steps=updates,batch_size=BATCH_SIZE,lr=LR,
                  weight_decay=WEIGHT_DECAY,clip_norm=CLIP_NORM,augmentations="independent-per-step matched D4",
                  dataset_sha256=hashlib.sha256((DATA_DIR/"train.jsonl").read_bytes()).hexdigest(),
                  question_order_sha256=schedule_digest,training_presentations=updates*BATCH_SIZE,
                  parameter_count=sum(p.numel() for p in model.parameters()),
                  per_task_presentations=Counter(train_data.tasks[i] for i in batches.ravel()),
                  profiled_flops_partial=next(s["profiled_flops_partial"] for s in specs if s["stem"]==stem),
                  environment=environment())
    settings["per_task_presentations"]=dict(settings["per_task_presentations"])
    (folder/"run.json").write_text(json.dumps(settings,indent=2),encoding="utf-8")
    return hist

def reevaluate_saved_stem(stem,seed,mode):
    """Evaluate saved end-to-end checkpoints on corrected selection S, without retraining."""
    eval_points=(750,1500,2250) if mode=="shape_only" else (5400,7200,9000)
    folder=OUT_DIR/"selection_stems"/mode/f"{stem}_seed{seed}"
    old_history=pd.read_csv(folder/"selection_history.csv").set_index("step")
    model=build_model(stem).to(DEVICE)
    hist=[]
    image_cache={}
    checkpoint_hashes={}
    for step in eval_points:
        path=folder/f"model_step{step}.pt"
        checkpoint=torch.load(path,map_location="cpu",weights_only=True)
        assert checkpoint["stem"]==stem and checkpoint["seed"]==seed and checkpoint["mode"]==mode and checkpoint["step"]==step
        model.load_state_dict(checkpoint["model"])
        metrics,preds=selection_evaluate(model,mode,image_cache)
        rec=dict(stem=stem,seed=seed,mode=mode,step=step,
                 train_loss=float(old_history.loc[step,"train_loss"]),**metrics)
        hist.append(rec)
        preds.drop(columns=["scene"],errors="ignore").to_csv(folder/f"selection_predictions_step{step}.csv",index=False)
        checkpoint_hashes[str(step)]=hashlib.sha256(path.read_bytes()).hexdigest()
        print("Reevaluated",mode,stem,seed,step,"Shape BA",round(metrics["shape_balanced_accuracy"],4),flush=True)
    pd.DataFrame(hist).to_csv(folder/"selection_history.csv",index=False)
    settings=json.loads((folder/"run.json").read_text(encoding="utf-8"))
    settings.pop("source_checkpoint_dir",None)
    settings.update(reevaluation_only=True,source_checkpoint_sha256_by_step=checkpoint_hashes,
                    evaluation_manifest_sha256=hashlib.sha256((OUT_DIR/"selection_and_confirmation_manifest.json").read_bytes()).hexdigest(),
                    profiled_flops_partial=next(a["profiled_flops_partial"] for a in specs if a["stem"]==stem),
                    evaluation_environment=environment())
    (folder/"run.json").write_text(json.dumps(settings,indent=2),encoding="utf-8")
    return hist

def saved_run_complete(stem,seed,mode):
    folder=OUT_DIR/"selection_stems"/mode/f"{stem}_seed{seed}"
    steps=(750,1500,2250) if mode=="shape_only" else (5400,7200,9000)
    return ((folder/"selection_history.csv").is_file() and (folder/"run.json").is_file()
            and all((folder/f"model_step{step}.pt").is_file() for step in steps))

histories=[]
for mode in ("shape_only","mixed"):
    for seed in SEEDS:
        for stem in STEMS:
            if RETRAIN_ALL_STEMS or not saved_run_complete(stem,seed,mode):
                histories+=train_stem(stem,seed,mode)
            else:
                histories+=reevaluate_saved_stem(stem,seed,mode)
summary=pd.DataFrame(histories)
summary.to_csv(OUT_DIR / "selection_stems/stem_selection_metrics.csv",index=False)
display(summary[summary.step.isin((2250,9000))].sort_values(["mode","seed","stem"]))

shape_only patchify 42 750 Shape BA 0.605 recolor both 0.498
shape_only patchify 42 1500 Shape BA 0.657 recolor both 0.58
shape_only patchify 42 2250 Shape BA 0.666 recolor both 0.578
shape_only pointwise 42 750 Shape BA 0.572 recolor both 0.466
shape_only pointwise 42 1500 Shape BA 0.615 recolor both 0.51
shape_only pointwise 42 2250 Shape BA 0.62 recolor both 0.532
shape_only conv_stem 42 750 Shape BA 0.573 recolor both 0.508
shape_only conv_stem 42 1500 Shape BA 0.659 recolor both 0.582
shape_only conv_stem 42 2250 Shape BA 0.728 recolor both 0.676
shape_only patchify 43 750 Shape BA 0.593 recolor both 0.498
shape_only patchify 43 1500 Shape BA 0.601 recolor both 0.496
shape_only patchify 43 2250 Shape BA 0.59 recolor both 0.49
shape_only pointwise 43 750 Shape BA 0.571 recolor both 0.504
shape_only pointwise 43 1500 Shape BA 0.631 recolor both 0.522
shape_only pointwise 43 2250 Shape BA 0.643 recolor both 0.556
shape_only conv_stem 43 750 Shape BA 0.67 recolor both 0.584
shape_only

,stem,seed,mode,step,train_loss,shape_balanced_accuracy,shape_recolor_both_correct,shape_recolor_both_correct_single_instance,color_balanced_accuracy,conjunction_bound_positive_correct,...,binding_auc_close,binding_auc_far,operator_three_case_joint,swap_eight_correct,conjunction_swap_flip_rate,swap_pairs,swap_eight_correct_heldout_created,conjunction_swap_flip_rate_heldout_created,swap_pairs_heldout_created,repaired_ood_binding_auc
35,conv_stem,42,mixed,9000,0.265643,0.747,0.670,0.621359,0.976000,0.816,...,0.790448,0.793520,0.606667,0.2275,0.2375,400.0,0.14,0.35,100.0,0.688184
29,patchify,42,mixed,9000,0.370535,0.666,0.578,0.368932,0.995333,0.852,...,0.649248,0.662368,0.518667,0.1200,0.1975,400.0,0.03,0.18,100.0,0.623036
32,pointwise,42,mixed,9000,0.362020,0.641,0.528,0.436893,0.998667,0.852,...,0.687280,0.702880,0.529333,0.1450,0.1950,400.0,0.08,0.20,100.0,0.658904
44,conv_stem,43,mixed,9000,0.331587,0.735,0.644,0.495146,0.994667,0.828,...,0.789552,0.779856,0.570667,0.1900,0.2950,400.0,0.14,0.26,100.0,0.557888
38,patchify,43,mixed,9000,0.444527,0.675,0.586,0.553398,0.992667,0.832,...,0.590848,0.615648,0.533333,0.1225,0.1650,400.0,0.07,0.24,100.0,0.464840
41,pointwise,43,mixed,9000,0.375435,0.678,0.594,0.417476,0.989333,0.800,...,0.699760,0.705936,0.548000,0.1475,0.1950,400.0,0.08,0.28,100.0,0.688900
53,conv_stem,44,mixed,9000,0.357888,0.758,0.678,0.524272,0.999333,0.872,...,0.555792,0.545920,0.552000,0.1000,0.1600,400.0,0.06,0.21,100.0,0.320656
47,patchify,44,mixed,9000,0.370430,0.679,0.584,0.427184,0.997333,0.888,...,0.660080,0.640240,0.542667,0.1050,0.1850,400.0,0.05,0.28,100.0,0.516924
50,pointwise,44,mixed,9000,0.418871,0.689,0.584,0.359223,0.996000,0.664,...,0.577464,0.591024,0.498667,0.0675,0.2300,400.0,0.03,0.28,100.0,0.342428
8,conv_stem,42,shape_only,2250,0.473331,0.728,0.676,0.436893,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Select a stem using prerequisite performance

D1 selects Conv Stem if its three-seed mean Shape balanced-accuracy gain over Patchify reaches five percentage points without meaningful Color or recoloring regressions. The rule is exploratory: later confirmation uses fresh training seeds and an independent evaluation set. No new stem design is selected using these results.

In [5]:
final=summary[(summary["mode"]=="mixed") & (summary.step==9000)].set_index(["seed","stem"])
p=final[["shape_balanced_accuracy","shape_recolor_both_correct","color_balanced_accuracy"]].unstack("stem")
base="patchify"
candidate="conv_stem"
delta={metric:(p[(metric,candidate)]-p[(metric,base)]) for metric in
       ("shape_balanced_accuracy","shape_recolor_both_correct","color_balanced_accuracy")}
shape_gain=float(delta["shape_balanced_accuracy"].mean())
recolor_gain=float(delta["shape_recolor_both_correct"].mean())
color_gain=float(delta["color_balanced_accuracy"].mean())
D1=bool(shape_gain>=.05 and recolor_gain>=-.02 and color_gain>=-.01)
print("Mean paired Shape gain:",shape_gain,"Recolouring:",recolor_gain,"Colour:",color_gain,"D1:",D1)

sigma=float(delta["shape_balanced_accuracy"].std(ddof=1))
sigma_planning=max(sigma,.04)  # conservative lower planning floor
# For five Holm-corrected two-sided tests, the first-step threshold is alpha/5 = 0.01;
# use t_{1-0.01/2} = t_0.995, NOT t_0.99.
n=5
while student_t.ppf(.995,n-1)*sigma_planning/np.sqrt(n)>.05 and n<100:n+=1
n_planned=min(n,8)  # explicit maximum; report insufficient precision if n > 8
print("Observed paired SD:",sigma,"Illustrative fresh confirmation seeds for ±5 pp half-width:",n)
record=dict(D1_selected=bool(D1),candidate=candidate if D1 else base,
            shape_gain=shape_gain,recolor_gain=recolor_gain,color_gain=color_gain,
            SESOI_shape=.05,paired_shape_sd=sigma,illustrative_confirmation_n=n,planned_confirmation_n=n_planned,
            confirmation_seed_cap=8,planning_sd_floor=.04,precision_goal_met_under_cap=(n<=8),
            seed_basis=list(SEEDS),selection_only=True)
(OUT_DIR/"selection_stems/D1_decision.json").write_text(json.dumps(record,indent=2),encoding="utf-8")
display(pd.DataFrame({k:v for k,v in delta.items()}))

Mean paired Shape gain: 0.07333333333333336 Recolouring: 0.08133333333333341 Colour: -0.00511111111111114 D1: True
Observed paired SD: 0.011590225767142557 Illustrative fresh confirmation seeds for ±5 pp half-width: 8


,shape_balanced_accuracy,shape_recolor_both_correct,color_balanced_accuracy
seed,,,
42,0.081,0.092,-0.019333
43,0.060,0.058,0.002000
44,0.079,0.094,0.002000
